# 2330 Aggressive Fill State Report

This notebook validates that `data/tw_stock_events/2330_20250909.npz` can produce actual fills in HftBacktest.

Test flow:

1. Load the converted Taiwan stock L2 event file.
2. Wait until a valid best bid and best ask are available.
3. Submit a GTC limit buy at the current best ask.
4. Submit a GTC limit sell at the current best bid.
5. Print state after each step: position, balance, fee, marked equity, trade count, trading value, and trading volume.

Assumptions: qty is in board lots, `CONTRACT_SIZE = 1000`, maker/taker fees are set to zero, and order latency is set to zero for this smoke test.

In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "tw_stock_events" / "2330_20250909.npz"

TICK_SIZE = 5.0
LOT_SIZE = 1.0
CONTRACT_SIZE = 1000.0
FILL_QTY = 1.0

DATA_FILE

WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_stock_events/2330_20250909.npz')

In [2]:
def import_hftbacktest_package(workspace_root: Path):
    root = workspace_root.resolve()
    original_path = list(sys.path)
    try:
        filtered_path = []
        for path_entry in original_path:
            if path_entry == "":
                continue
            try:
                if Path(path_entry).resolve() == root:
                    continue
            except Exception:
                pass
            filtered_path.append(path_entry)
        sys.path = filtered_path
        sys.modules.pop("hftbacktest", None)
        module = importlib.import_module("hftbacktest")
        if not hasattr(module, "BacktestAsset"):
            raise ImportError(f"loaded {module!r}, but BacktestAsset is missing")
        return module
    finally:
        sys.path = original_path

hbtpkg = import_hftbacktest_package(ROOT)
print("kernel executable:", sys.executable)
print("hftbacktest:", getattr(hbtpkg, "__version__", "unknown"))
print("package file:", getattr(hbtpkg, "__file__", None))

kernel executable: C:\Users\zoufuc\AppData\Local\Programs\Python\Python311\python.exe
hftbacktest: 2.4.4
package file: C:\Users\zoufuc\AppData\Local\Programs\Python\Python311\Lib\site-packages\hftbacktest\__init__.py


In [3]:
def build_backtest():
    asset = (
        hbtpkg.BacktestAsset()
        .data(str(DATA_FILE))
        .linear_asset(CONTRACT_SIZE)
        .constant_order_latency(0, 0)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .trading_value_fee_model(0.0, 0.0)
        .tick_size(TICK_SIZE)
        .lot_size(LOT_SIZE)
        .last_trades_capacity(100)
    )
    return hbtpkg.HashMapMarketDepthBacktest([asset])


def state_snapshot(hbt, asset_no=0):
    depth = hbt.depth(asset_no)
    state = hbt.state_values(asset_no)
    bid = float(depth.best_bid)
    ask = float(depth.best_ask)
    mark = (bid + ask) / 2.0 if np.isfinite(bid) and np.isfinite(ask) else 0.0
    position = float(state.position)
    balance = float(state.balance)
    fee = float(state.fee)
    equity = balance + position * mark * CONTRACT_SIZE - fee
    return {
        "bid": bid,
        "ask": ask,
        "mark": mark,
        "position": position,
        "balance": balance,
        "fee": fee,
        "equity": equity,
        "num_trades": int(state.num_trades),
        "trading_value": float(state.trading_value),
        "trading_volume": float(state.trading_volume),
    }


def print_state(label, order_id, snap):
    order_text = "" if order_id is None else f" order_id={order_id}"
    print(
        f"{label:<18}{order_text:<14}"
        f" bid={snap['bid']:.2f}"
        f" ask={snap['ask']:.2f}"
        f" mark={snap['mark']:.2f}"
        f" pos={snap['position']:.4f}"
        f" balance={snap['balance']:.2f}"
        f" fee={snap['fee']:.2f}"
        f" equity={snap['equity']:.2f}"
        f" trades={snap['num_trades']}"
        f" value={snap['trading_value']:.2f}"
        f" volume={snap['trading_volume']:.4f}"
    )


def wait_for_bbo(hbt, asset_no=0):
    for _ in range(10):
        result = hbt.elapse(1_000_000_000)
        if result != 0:
            raise RuntimeError("Backtest ended before BBO is available")
        depth = hbt.depth(asset_no)
        if np.isfinite(depth.best_bid) and np.isfinite(depth.best_ask):
            return
    raise RuntimeError("No valid BBO found")

In [4]:
def submit_and_report(hbt, side, order_id, px, qty=FILL_QTY, asset_no=0):
    before = state_snapshot(hbt, asset_no)
    print_state(f"before_{side}", order_id, before)

    if side == "buy":
        rc = hbt.submit_buy_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
    elif side == "sell":
        rc = hbt.submit_sell_order(asset_no, order_id, px, qty, hbtpkg.GTC, hbtpkg.LIMIT, False)
    else:
        raise ValueError(side)
    print(f"submit_{side:<11} order_id={order_id:<6} px={px:.2f} qty={qty:.4f} rc={rc}")

    response = hbt.wait_order_response(asset_no, order_id, 10_000_000)
    after = state_snapshot(hbt, asset_no)
    print(f"response_{side:<9} order_id={order_id:<6} response={response}")
    print_state(f"after_{side}", order_id, after)

    assert after["num_trades"] == before["num_trades"] + 1, f"{side} did not fill"
    hbt.clear_inactive_orders(asset_no)
    print_state(f"clear_{side}", order_id, state_snapshot(hbt, asset_no))


def run_once():
    hbt = build_backtest()
    try:
        wait_for_bbo(hbt)
        print_state("initial_bbo", None, state_snapshot(hbt))

        depth = hbt.depth(0)
        print("\naggressive buy at best ask")
        submit_and_report(hbt, "buy", 30_001, float(depth.best_ask))

        depth = hbt.depth(0)
        print("\naggressive sell at best bid")
        submit_and_report(hbt, "sell", 30_002, float(depth.best_bid))

        print("\nfinal")
        final_state = state_snapshot(hbt)
        print_state("final_state", None, final_state)
        assert final_state["position"] == 0.0
        return final_state
    finally:
        hbt.close()

final_state = run_once()
final_state

initial_bbo                      bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000

aggressive buy at best ask
before_buy         order_id=30001 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=0.00 fee=0.00 equity=0.00 trades=0 value=0.00 volume=0.0000
submit_buy         order_id=30001  px=1195.00 qty=1.0000 rc=0
response_buy       order_id=30001  response=0
after_buy          order_id=30001 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000


clear_buy          order_id=30001 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000

aggressive sell at best bid
before_sell        order_id=30002 bid=1190.00 ask=1195.00 mark=1192.50 pos=1.0000 balance=-1195000.00 fee=0.00 equity=-2500.00 trades=1 value=1195000.00 volume=1.0000
submit_sell        order_id=30002  px=1190.00 qty=1.0000 rc=0
response_sell      order_id=30002  response=0
after_sell         order_id=30002 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=-5000.00 fee=0.00 equity=-5000.00 trades=2 value=2385000.00 volume=2.0000
clear_sell         order_id=30002 bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=-5000.00 fee=0.00 equity=-5000.00 trades=2 value=2385000.00 volume=2.0000

final
final_state                      bid=1190.00 ask=1195.00 mark=1192.50 pos=0.0000 balance=-5000.00 fee=0.00 equity=-5000.00 trades=2 value=2385000.00 volume=2.0000


{'bid': 1190.0,
 'ask': 1195.0,
 'mark': 1192.5,
 'position': 0.0,
 'balance': -5000.0,
 'fee': 0.0,
 'equity': -5000.0,
 'num_trades': 2,
 'trading_value': 2385000.0,
 'trading_volume': 2.0}